<a href="https://colab.research.google.com/github/Linford24/AfricaBp_Computer_Vision_for_Biodiversity_Classification/blob/main/variant_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update && apt-get install -y fastqc trimmomatic seqkit bwa samtools
!pip install httpx tqdm pandas multiqc

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fastqc is already the newest version (0.11.9+dfsg-5).
trimmomatic is already the newest versio

Downloading The Fasta Files

In [ ]:
import asyncio
from pathlib import Path
import time
import httpx
from tqdm.asyncio import tqdm

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

URLS = [
    # SRR2589044
    (
        "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/004/SRR2589044/SRR2589044_1.fastq.gz",
        RAW_DIR / "SRR2589044_1.fastq.gz",
    ),
    (
        "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/004/SRR2589044/SRR2589044_2.fastq.gz",
        RAW_DIR / "SRR2589044_2.fastq.gz",
    ),
    # SRR2584863
    (
        "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/003/SRR2584863/SRR2584863_1.fastq.gz",
        RAW_DIR / "SRR2584863_1.fastq.gz",
    ),
    (
        "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/003/SRR2584863/SRR2584863_2.fastq.gz",
        RAW_DIR / "SRR2584863_2.fastq.gz",
    ),
    # SRR2584866
    (
        "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/006/SRR2584866/SRR2584866_1.fastq.gz",
        RAW_DIR / "SRR2584866_1.fastq.gz",
    ),
    (
        "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR258/006/SRR2584866/SRR2584866_2.fastq.gz",
        RAW_DIR / "SRR2584866_2.fastq.gz",
    ),
]


async def download_file(
    client: httpx.AsyncClient,
    url: str,
    output_path: Path,
    retries: int = 5,
    delay: int = 5,
):
    # Skip if non-empty file already exists (-s check)
    if output_path.exists() and output_path.stat().st_size > 0:
        print(f"[SKIP] {output_path.name} already exists.")
        return

    for attempt in range(1, retries + 1):
        try:
            async with client.stream("GET", url, follow_redirects=True) as response:
                response.raise_for_status()
                total = int(response.headers.get("content-length", 0))

                with (
                    open(output_path, "wb") as f,
                    tqdm(
                        total=total,
                        unit="iB",
                        unit_scale=True,
                        desc=output_path.name,
                        leave=True,
                    ) as pbar,
                ):
                    async for chunk in response.aiter_bytes(chunk_size=1024 * 1024):  # 1MB chunks
                        f.write(chunk)
                        pbar.update(len(chunk))
            break
        except Exception as e:
            if output_path.exists():
                output_path.unlink()  # Clean up partial download
            if attempt < retries:
                await asyncio.sleep(delay)
            else:
                raise RuntimeError(
                    f"Failed to download {output_path.name} after {retries} retries: {e}"
                )


async def main():
    print("==========================================")
    print("Downloading LTEE Ara-3 sequencing reads")
    print(f"Started: {time.ctime()}")
    print("==========================================")

    # Allow up to 3 concurrent downloads, adjust timeout for large FASTQ files
    limits = httpx.Limits(max_connections=5, max_keepalive_connections=5)
    timeout = httpx.Timeout(60.0, connect=10.0)

    async with httpx.AsyncClient(limits=limits, timeout=timeout) as client:
        # Download files concurrently using asyncio.gather
        tasks = [download_file(client, url, path) for url, path in URLS]
        await asyncio.gather(*tasks)

    print("\n==========================================")
    print("Download complete")
    print(f"Finished: {time.ctime()}")
    print("==========================================")

    for file in sorted(RAW_DIR.iterdir()):
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"{file.name:<30} {size_mb:.2f} MB")


# Run directly in Jupyter Notebook (Jupyter already has an event loop running)
await main()

Started: Tue Aug 11 22:35:03 2026
[SKIP] SRR2589044_1.fastq.gz already exists.
[SKIP] SRR2589044_2.fastq.gz already exists.
[SKIP] SRR2584863_1.fastq.gz already exists.
[SKIP] SRR2584863_2.fastq.gz already exists.
[SKIP] SRR2584866_1.fastq.gz already exists.
[SKIP] SRR2584866_2.fastq.gz already exists.

Download complete
Finished: Tue Aug 11 22:35:03 2026
SRR2584863_1.fastq.gz          174.80 MB
SRR2584863_2.fastq.gz          182.24 MB
SRR2584866_1.fastq.gz          308.89 MB
SRR2584866_2.fastq.gz          295.92 MB
SRR2589044_1.fastq.gz          123.16 MB
SRR2589044_2.fastq.gz          128.00 MB


In [ ]:
from pathlib import Path
import subprocess
import time

RAW_DIR = Path("data/raw")
FASTQC_DIR = Path("results/fastqc_raw")
MULTIQC_DIR = Path("results/multiqc/raw")

THREADS = 4

FASTQC_DIR.mkdir(parents=True, exist_ok=True)
MULTIQC_DIR.mkdir(parents=True, exist_ok=True)

print("==========================================")
print("Raw FASTQ quality control")
print(f"Started: {time.ctime()}")
print("==========================================")

# Gather all .fastq.gz files in RAW_DIR
fastq_files = list(RAW_DIR.glob("*.fastq.gz"))

if not fastq_files:
    raise FileNotFoundError(f"No .fastq.gz files found in {RAW_DIR}")

# 1. Run FastQC
fastqc_cmd = [
    "fastqc",
    "--threads",
    str(THREADS),
    "--outdir",
    str(FASTQC_DIR),
] + [str(f) for f in fastq_files]

print("Running FastQC...")
subprocess.run(fastqc_cmd, check=True)

# 2. Run MultiQC
print("\nRunning MultiQC...")
multiqc_cmd = [
    "multiqc",
    str(FASTQC_DIR),
    "--outdir",
    str(MULTIQC_DIR),
    "--force",
]

subprocess.run(multiqc_cmd, check=True)

print("\n==========================================")
print("Raw-read QC complete")
print(f"Finished: {time.ctime()}")
print("\nMultiQC report:")
print(MULTIQC_DIR / "multiqc_report.html")
print("==========================================")

Raw FASTQ quality control
Started: Tue Aug 11 22:35:09 2026
Running FastQC...

Running MultiQC...

Raw-read QC complete
Finished: Tue Aug 11 22:37:05 2026

MultiQC report:
results/multiqc/raw/multiqc_report.html


In [ ]:
import gzip
import os
from pathlib import Path
import shutil
import subprocess
import time
import pandas as pd
import requests

# Directories
METADATA_PATH = Path("metadata/samples.tsv")
TRIM_DIR = Path("data/trimmed")
UNPAIRED_DIR = TRIM_DIR / "unpaired"
STATS_DIR = Path("results/statistics/trimming")

THREADS = 4

# Create directories
TRIM_DIR.mkdir(parents=True, exist_ok=True)
UNPAIRED_DIR.mkdir(parents=True, exist_ok=True)
STATS_DIR.mkdir(parents=True, exist_ok=True)

print("==============================================")
print("LTEE Ara-3 read trimming")
print(f"Started: {time.ctime()}")
print("==============================================")

# ----------------------------------------------------------
# Locate or Fallback-Download the Nextera adapter file
# ----------------------------------------------------------
adapters_file = None

# Known system locations where apt-get, conda, and brew store adapters
search_paths = [
    Path("/usr/share/trimmomatic"),
    Path("/usr/share/seq-answers"),
    Path("/usr/local/share/trimmomatic"),
    Path("/etc/trimmomatic"),
]

conda_prefix = os.environ.get("CONDA_PREFIX")
if conda_prefix:
    search_paths.insert(0, Path(conda_prefix))

for base_path in search_paths:
    if base_path.exists():
        found = list(base_path.rglob("NexteraPE-PE.fa"))
        if found:
            adapters_file = found[0]
            break

# Fallback: Download NexteraPE-PE.fa locally if not found on the system
if not adapters_file or not adapters_file.exists():
    fallback_adapter_dir = Path("metadata/adapters")
    fallback_adapter_dir.mkdir(parents=True, exist_ok=True)
    adapters_file = fallback_adapter_dir / "NexteraPE-PE.fa"

    if not adapters_file.exists():
        print(
            "NexteraPE-PE.fa not found in system paths. Downloading official adapter file..."
        )
        url = "https://raw.githubusercontent.com/usadellab/Trimmomatic/main/adapters/NexteraPE-PE.fa"
        with requests.get(url, stream=True, timeout=30) as resp:
            resp.raise_for_status()
            with open(adapters_file, "wb") as f:
                for chunk in resp.iter_content(chunk_size=1024 * 256):
                    f.write(chunk)

print(f"Using adapter file:\n{adapters_file}\n")

# ----------------------------------------------------------
# Locate CLI tools
# ----------------------------------------------------------
trimmomatic_bin = (
    shutil.which("TrimmomaticPE")
    or shutil.which("trimmomatic")
    or "TrimmomaticPE"
)
seqkit_bin = shutil.which("seqkit")

# The apt-installed wrapper scripts (TrimmomaticPE / TrimmomaticSE) are
# already mode-specific and do NOT accept a "PE"/"SE" argument, unlike the
# newer unified `trimmomatic` CLI (e.g. from conda), which requires it.
bin_name = Path(str(trimmomatic_bin)).name
if bin_name in ("TrimmomaticPE", "TrimmomaticSE"):
    mode_args = []
else:
    mode_args = ["PE"]

print(f"Using Trimmomatic binary: {trimmomatic_bin}")
print(f"Mode args: {mode_args if mode_args else '(none needed - wrapper script)'}\n")

# ----------------------------------------------------------
# Process each paired-end sample
# ----------------------------------------------------------
df_samples = pd.read_csv(METADATA_PATH, sep="\t")

for _, row in df_samples.iterrows():
    sample = row["sample"]
    generation = row["generation"]
    run = row["run"]
    r1 = row["R1"]
    r2 = row["R2"]

    print("----------------------------------------------")
    print(f"Sample:     {sample}")
    print(f"Generation: {generation}")
    print(f"Run:        {run}")
    print("----------------------------------------------")

    # Sanity-check inputs exist and are non-empty before we bother calling
    # Trimmomatic, since a missing/truncated input is a common silent
    # cause of an immediate exit-1.
    for input_path in (r1, r2):
        p = Path(input_path)
        if not p.exists():
            raise FileNotFoundError(f"Input read file not found: {p}")
        if p.stat().st_size == 0:
            raise ValueError(f"Input read file is empty: {p}")

    out_r1 = TRIM_DIR / f"{run}_1.trim.fastq.gz"
    out_r2 = TRIM_DIR / f"{run}_2.trim.fastq.gz"
    out_u1 = UNPAIRED_DIR / f"{run}_1.unpaired.fastq.gz"
    out_u2 = UNPAIRED_DIR / f"{run}_2.unpaired.fastq.gz"
    summary = STATS_DIR / f"{run}.trimmomatic_summary.txt"

    # Skip if outputs already exist and are non-empty
    if (
        out_r1.exists()
        and out_r1.stat().st_size > 0
        and out_r2.exists()
        and out_r2.stat().st_size > 0
    ):
        print(f"[SKIP] Trimmed paired reads already exist for {run}\n")
        continue

    trimmomatic_cmd = [
        trimmomatic_bin,
        *mode_args,
        "-threads",
        str(THREADS),
        "-phred33",
        "-summary",
        str(summary),
        str(r1),
        str(r2),
        str(out_r1),
        str(out_u1),
        str(out_r2),
        str(out_u2),
        f"ILLUMINACLIP:{adapters_file}:2:40:15",
        "SLIDINGWINDOW:4:20",
        "MINLEN:25",
    ]

    # Capture stdout/stderr so a failure actually tells us why, instead of
    # just raising a bare CalledProcessError with no diagnostic info.
    result = subprocess.run(trimmomatic_cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)

    if result.returncode != 0:
        print(f"[FAILED] Trimmomatic exited with status {result.returncode} for {run}")
        result.check_returncode()  # raises CalledProcessError with same semantics as check=True

    print()

# ----------------------------------------------------------
# Check compressed FASTQ integrity
# ----------------------------------------------------------
print("Checking trimmed FASTQ integrity...")
trimmed_files = list(TRIM_DIR.glob("*.trim.fastq.gz"))

for filepath in trimmed_files:
    try:
        with gzip.open(filepath, "rb") as f:
            while f.read(1024 * 1024):  # Read in 1MB chunks
                pass
        print(f"[OK] {filepath}")
    except Exception as e:
        print(f"[CORRUPT] {filepath}: {e}")

# ----------------------------------------------------------
# Generate Read Statistics
# ----------------------------------------------------------
print("\nGenerating trimmed-read statistics...")
stats_out_file = STATS_DIR / "trimmed_seqkit_stats.txt"

if seqkit_bin:
    seqkit_cmd = [seqkit_bin, "stats"] + [str(f) for f in trimmed_files]
    with open(stats_out_file, "w") as out:
        subprocess.run(seqkit_cmd, stdout=out, check=True)
else:
    # Pure Python fallback for FASTQ stats if seqkit is absent
    with open(stats_out_file, "w") as out_f:
        out_f.write("file\tnum_seqs\tsum_len\n")
        for filepath in sorted(trimmed_files):
            num_reads = 0
            total_bases = 0
            with gzip.open(filepath, "rt") as f:
                for i, line in enumerate(f):
                    if i % 4 == 1:
                        num_reads += 1
                        total_bases += len(line.strip())
            out_f.write(f"{filepath.name}\t{num_reads}\t{total_bases}\n")

print("\n==============================================")
print("Trimming complete")
print(f"Finished: {time.ctime()}")
print(f"\nPaired reads:\n{TRIM_DIR}")
print(f"\nStatistics:\n{STATS_DIR}")
print("==============================================")

LTEE Ara-3 read trimming
Started: Tue Aug 11 22:38:49 2026
Using adapter file:
/usr/share/trimmomatic/NexteraPE-PE.fa

Using Trimmomatic binary: /usr/bin/TrimmomaticPE
Mode args: (none needed - wrapper script)

----------------------------------------------
Sample:     Ara3_5000
Generation: 5000
Run:        SRR2589044
----------------------------------------------
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

TrimmomaticPE: Started with arguments:
 -threads 4 -phred33 -summary results/statistics/trimming/SRR2589044.trimmomatic_summary.txt data/raw/SRR2589044_1.fastq.gz data/raw/SRR2589044_2.fastq.gz data/trimmed/SRR2589044_1.trim.fastq.gz data/trimmed/unpaired/SRR2589044_1.unpaired.fastq.gz data/trimmed/SRR

In [ ]:
if not trimmed_files:
    raise FileNotFoundError(f"No .trim.fastq.gz files found in {TRIM_DIR}")

# Locate binaries in environment
fastqc_bin = shutil.which("fastqc")
multiqc_bin = shutil.which("multiqc")

# 1. Run FastQC
print("Running FastQC...")
if fastqc_bin:
    fastqc_cmd = [
        fastqc_bin,
        "--threads",
        str(THREADS),
        "--outdir",
        str(FASTQC_DIR),
    ] + [str(f) for f in trimmed_files]
    subprocess.run(fastqc_cmd, check=True)
else:
    # Fallback for Jupyter environment PATH resolution
    files_str = " ".join(f'"{f}"' for f in trimmed_files)
    get_ipython().system(
        f'fastqc --threads {THREADS} --outdir "{FASTQC_DIR}" {files_str}'
    )

# 2. Run MultiQC
print("\nRunning MultiQC...")
if multiqc_bin:
    multiqc_cmd = [
        multiqc_bin,
        str(FASTQC_DIR),
        "--outdir",
        str(MULTIQC_DIR),
        "--force",
    ]
    subprocess.run(multiqc_cmd, check=True)
else:
    # Fallback for Jupyter environment PATH resolution
    get_ipython().system(
        f'multiqc "{FASTQC_DIR}" --outdir "{MULTIQC_DIR}" --force'
    )

print("\n==============================================")
print("Post-trimming QC complete")
print(f"Finished: {time.ctime()}")
print("\nMultiQC report:")
print(MULTIQC_DIR / "multiqc_report.html")
print("==============================================")

Running FastQC...

Running MultiQC...

Post-trimming QC complete
Finished: Tue Aug 11 22:48:45 2026

MultiQC report:
results/multiqc/raw/multiqc_report.html


In [ ]:
import asyncio
import gzip
import os
from pathlib import Path
import shutil
import subprocess
import time
import httpx
from tqdm.asyncio import tqdm

# Directories and Paths
REF_DIR = Path("data/reference")
STATS_DIR = Path("results/statistics/reference")

REF_GZ = REF_DIR / "ecoli_rel606.fasta.gz"
REF = REF_DIR / "ecoli_rel606.fasta"

REF_URL = "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/000/017/985/GCA_000017985.1_ASM1798v1/GCA_000017985.1_ASM1798v1_genomic.fna.gz"

# Create directories
REF_DIR.mkdir(parents=True, exist_ok=True)
STATS_DIR.mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------
# High-speed HTTPX Async Downloader
# ----------------------------------------------------------
async def async_download(
    url: str, output_path: Path, retries: int = 5, delay: int = 5
):
    timeout = httpx.Timeout(60.0, connect=10.0)
    async with httpx.AsyncClient(
        timeout=timeout, follow_redirects=True
    ) as client:
        for attempt in range(1, retries + 1):
            try:
                async with client.stream("GET", url) as response:
                    response.raise_for_status()
                    total = int(response.headers.get("content-length", 0))

                    with (
                        open(output_path, "wb") as f,
                        tqdm(
                            total=total,
                            unit="iB",
                            unit_scale=True,
                            desc=output_path.name,
                        ) as pbar,
                    ):
                        async for chunk in response.aiter_bytes(
                            chunk_size=1024 * 1024
                        ):  # 1MB chunks
                            f.write(chunk)
                            pbar.update(len(chunk))
                break
            except Exception as e:
                if output_path.exists():
                    output_path.unlink()
                if attempt < retries:
                    print(
                        f"  [Attempt {attempt} failed]: {e}. Retrying in {delay}s..."
                    )
                    await asyncio.sleep(delay)
                else:
                    raise RuntimeError(
                        f"Failed to download {url} after {retries} retries."
                    )


async def main():
    print("==============================================")
    print("Preparing E. coli REL606 reference genome")
    print(f"Started: {time.ctime()}")
    print("==============================================")

    # ----------------------------------------------------------
    # 1. Download reference
    # ----------------------------------------------------------
    if REF.exists() and REF.stat().st_size > 0:
        print(f"[SKIP] Uncompressed reference already exists:\n{REF}")
    elif REF_GZ.exists() and REF_GZ.stat().st_size > 0:
        print(f"[FOUND] Compressed reference already exists:\n{REF_GZ}")
    else:
        print(f"[DOWNLOAD] REL606 reference genome via HTTPX...")
        await async_download(REF_URL, REF_GZ)

    # ----------------------------------------------------------
    # 2. Test compressed file integrity
    # ----------------------------------------------------------
    if REF_GZ.exists() and REF_GZ.stat().st_size > 0 and not REF.exists():
        print("\nTesting gzip integrity...")
        try:
            with gzip.open(REF_GZ, "rb") as gz_f:
                while gz_f.read(1024 * 1024):
                    pass
            print("[OK] Reference archive passed gzip check.")
        except Exception as e:
            raise RuntimeError(
                f"[CORRUPT] Reference archive failed gzip check: {e}"
            )

    # ----------------------------------------------------------
    # 3. Uncompress reference
    # ----------------------------------------------------------
    if not REF.exists() or REF.stat().st_size == 0:
        print("\nUncompressing reference...")
        with gzip.open(REF_GZ, "rb") as f_in:
            with open(REF, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)
        print(f"[OK] Uncompressed to {REF}")

    # ----------------------------------------------------------
    # 4. Inspect reference
    # ----------------------------------------------------------
    print("\nReference sequence header:")
    with open(REF, "r") as f:
        for line in f:
            if line.startswith(">"):
                print(line.strip())

    # ----------------------------------------------------------
    # 5. Reference statistics
    # ----------------------------------------------------------
    print("\nCalculating reference statistics...")
    seqkit_bin = shutil.which("seqkit")
    stats_file = STATS_DIR / "REL606_seqkit_stats.txt"

    if seqkit_bin:
        seqkit_cmd = [seqkit_bin, "stats", str(REF)]
        with open(stats_file, "w") as out:
            subprocess.run(seqkit_cmd, stdout=out, check=True)
    else:
        with open(REF, "r") as f, open(stats_file, "w") as out:
            num_seqs = 0
            total_len = 0
            for line in f:
                if line.startswith(">"):
                    num_seqs += 1
                else:
                    total_len += len(line.strip())
            stats_text = (
                f"file\tformat\ttype\tnum_seqs\tsum_len\n"
                f"{REF.name}\tFASTA\tDNA\t{num_seqs}\t{total_len}\n"
            )
            out.write(stats_text)

    with open(stats_file, "r") as f:
        print(f.read())

    # ----------------------------------------------------------
    # 6. BWA index
    # ----------------------------------------------------------
    bwt_index = Path(f"{REF}.bwt")
    bwa_bin = shutil.which("bwa") or "bwa"

    if not bwt_index.exists():
        print("\nBuilding BWA index...")
        subprocess.run([bwa_bin, "index", str(REF)], check=True)
    else:
        print("\n[SKIP] BWA index already present.")

    # ----------------------------------------------------------
    # 7. samtools FASTA index
    # ----------------------------------------------------------
    fai_index = Path(f"{REF}.fai")
    samtools_bin = shutil.which("samtools") or "samtools"

    if not fai_index.exists():
        print("\nBuilding samtools FASTA index...")
        subprocess.run([samtools_bin, "faidx", str(REF)], check=True)
    else:
        print("\n[SKIP] samtools FASTA index already present.")

    # ----------------------------------------------------------
    # 8. Final check
    # ----------------------------------------------------------
    print("\nReference directory:")
    for file in sorted(REF_DIR.iterdir()):
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"{file.name:<35} {size_mb:.2f} MB")

    print("\n==============================================")
    print("Reference preparation complete")
    print(f"Finished: {time.ctime()}")
    print("==============================================")


# Run directly inside Jupyter cell
await main()

Preparing E. coli REL606 reference genome
Started: Tue Aug 11 22:54:29 2026
[SKIP] Uncompressed reference already exists:
data/reference/ecoli_rel606.fasta

Reference sequence header:
>CP000819.1 Escherichia coli B str. REL606, complete genome

Calculating reference statistics...
file                               format  type  num_seqs    sum_len    min_len    avg_len    max_len
data/reference/ecoli_rel606.fasta  FASTA   DNA          1  4,629,812  4,629,812  4,629,812  4,629,812


[SKIP] BWA index already present.

Building samtools FASTA index...

Reference directory:
ecoli_rel606.fasta                  4.47 MB
ecoli_rel606.fasta.amb              0.00 MB
ecoli_rel606.fasta.ann              0.00 MB
ecoli_rel606.fasta.bwt              4.42 MB
ecoli_rel606.fasta.fai              0.00 MB
ecoli_rel606.fasta.gz               1.31 MB
ecoli_rel606.fasta.pac              1.10 MB
ecoli_rel606.fasta.sa               2.21 MB

Reference preparation complete
Finished: Tue Aug 11 22:54:29 2026


**Alignment And Coverage Assessment**

In [ ]:
import csv
from pathlib import Path
import shutil
import subprocess
import time
import pandas as pd

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------
METADATA_PATH = Path("metadata/samples.tsv")
REF = Path("data/reference/ecoli_rel606.fasta")
REF_FAI = Path("data/reference/ecoli_rel606.fasta.fai")
TRIM_DIR = Path("data/trimmed")
BAM_DIR = Path("results/bam")
QC_DIR = Path("results/statistics/alignment")

THREADS = 4

BAM_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)

# Binary discovery
bwa_bin = shutil.which("bwa") or "bwa"
samtools_bin = shutil.which("samtools") or "samtools"

# ----------------------------------------------------------
# Check required reference files
# ----------------------------------------------------------
if not REF.exists() or REF.stat().st_size == 0:
    raise FileNotFoundError(f"Reference genome not found: {REF}")

if not REF_FAI.exists() or REF_FAI.stat().st_size == 0:
    raise FileNotFoundError(f"samtools reference index missing: {REF_FAI}")

# Calculate genome size from .fai
genome_size = 0
with open(REF_FAI, "r") as f:
    for line in f:
        fields = line.strip().split("\t")
        if len(fields) >= 2:
            genome_size += int(fields[1])

print("==============================================")
print("LTEE Ara-3 alignment")
print(f"Started: {time.ctime()}")
print()
print(f"Reference:   {REF}")
print(f"Genome size: {genome_size:,} bp")
print(f"Threads:     {THREADS}")
print("==============================================")

# ----------------------------------------------------------
# Initialize Summary Table
# ----------------------------------------------------------
SUMMARY_PATH = QC_DIR / "alignment_summary.tsv"
summary_headers = [
    "sample",
    "generation",
    "run",
    "total_reads",
    "mapped_reads",
    "mapping_percent",
    "properly_paired",
    "properly_paired_percent",
    "mean_depth",
    "breadth_1x_percent",
    "breadth_10x_percent",
]

# Create or overwrite summary file with headers
with open(SUMMARY_PATH, "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(summary_headers)

# ----------------------------------------------------------
# Helper Functions
# ----------------------------------------------------------
def run_count(cmd):
    """Executes a command and returns the integer output (e.g., samtools view -c)."""
    res = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return int(res.stdout.strip())


def calculate_depth_stats(depth_file):
    """Parses depth file output to compute mean depth, 1x breadth, and 10x breadth."""
    total_positions = 0
    sum_depth = 0
    covered_1x = 0
    covered_10x = 0

    with open(depth_file, "r") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 3:
                depth = int(parts[2])
                total_positions += 1
                sum_depth += depth
                if depth >= 1:
                    covered_1x += 1
                if depth >= 10:
                    covered_10x += 1

    if total_positions == 0:
        return 0.0, 0.0, 0.0

    mean_depth = sum_depth / total_positions
    breadth_1x = (covered_1x / total_positions) * 100
    breadth_10x = (covered_10x / total_positions) * 100

    return mean_depth, breadth_1x, breadth_10x


# ----------------------------------------------------------
# Process Samples
# ----------------------------------------------------------
df_samples = pd.read_csv(METADATA_PATH, sep="\t")

for _, row in df_samples.iterrows():
    sample = str(row["sample"])
    generation = str(row["generation"])
    run = str(row["run"])

    print("\n==============================================")
    print(f"Sample:     {sample}")
    print(f"Generation: {generation}")
    print(f"Run:        {run}")
    print("==============================================")

    r1 = TRIM_DIR / f"{run}_1.trim.fastq.gz"
    r2 = TRIM_DIR / f"{run}_2.trim.fastq.gz"

    bam = BAM_DIR / f"{run}.aligned.sorted.bam"
    flagstat = QC_DIR / f"{run}.flagstat.txt"
    stats = QC_DIR / f"{run}.samtools_stats.txt"
    depth = QC_DIR / f"{run}.depth.txt"

    # Check input files
    if not r1.exists() or r1.stat().st_size == 0:
        raise FileNotFoundError(f"Missing R1 read file: {r1}")
    if not r2.exists() or r2.stat().st_size == 0:
        raise FileNotFoundError(f"Missing R2 read file: {r2}")

    # 1. Alignment (bwa mem | samtools sort)
    if bam.exists() and bam.stat().st_size > 0:
        print(f"[SKIP] BAM already exists:\n{bam}")
    else:
        print("Aligning reads with BWA-MEM...")
        rg_header = f"@RG\\tID:{run}\\tSM:{sample}\\tPL:ILLUMINA"

        cmd_bwa = [
            bwa_bin,
            "mem",
            "-t",
            str(THREADS),
            "-R",
            rg_header,
            str(REF),
            str(r1),
            str(r2),
        ]

        cmd_sort = [
            samtools_bin,
            "sort",
            "-@",
            str(THREADS),
            "-o",
            str(bam),
            "-",
        ]

        # Pipe bwa stdout directly into samtools sort stdin
        p1 = subprocess.Popen(cmd_bwa, stdout=subprocess.PIPE)
        p2 = subprocess.Popen(cmd_sort, stdin=p1.stdout, stdout=subprocess.PIPE)
        p1.stdout.close()  # Allow p1 to receive SIGPIPE if p2 exits
        p2.communicate()

        if p2.returncode != 0:
            raise RuntimeError(f"Alignment/Sorting failed for sample {run}")

    # 2. BAM Integrity Check
    print("Checking BAM integrity...")
    subprocess.run([samtools_bin, "quickcheck", "-v", str(bam)], check=True)
    print("[OK] BAM passed samtools quickcheck.")

    # 3. Index BAM
    bam_index = Path(f"{bam}.bai")
    if not bam_index.exists() or bam_index.stat().st_size == 0:
        print("Indexing BAM...")
        subprocess.run(
            [samtools_bin, "index", "-@", str(THREADS), str(bam)], check=True
        )
    else:
        print("[SKIP] BAM index already exists.")

    # 4. Alignment Statistics
    print("Generating flagstat...")
    with open(flagstat, "w") as out:
        subprocess.run(
            [samtools_bin, "flagstat", "-@", str(THREADS), str(bam)],
            stdout=out,
            check=True,
        )

    print("Generating samtools stats...")
    with open(stats, "w") as out:
        subprocess.run(
            [samtools_bin, "stats", "-@", str(THREADS), str(bam)],
            stdout=out,
            check=True,
        )

    # 5. Coverage Statistics
    print("Calculating genome-wide depth...")
    with open(depth, "w") as out:
        subprocess.run(
            [samtools_bin, "depth", "-aa", str(bam)], stdout=out, check=True
        )

    # 6. Extract Summary Values
    total_reads = run_count([samtools_bin, "view", "-c", "-F", "0x900", str(bam)])
    mapped_reads = run_count([samtools_bin, "view", "-c", "-F", "0x904", str(bam)])
    proper_pairs = run_count(
        [samtools_bin, "view", "-c", "-f", "0x2", "-F", "0x900", str(bam)]
    )

    mapping_pct = (mapped_reads / total_reads * 100) if total_reads > 0 else 0.0
    proper_pct = (proper_pairs / total_reads * 100) if total_reads > 0 else 0.0

    mean_depth, breadth_1x, breadth_10x = calculate_depth_stats(depth)

    # Append row to TSV
    row_data = [
        sample,
        generation,
        run,
        total_reads,
        mapped_reads,
        f"{mapping_pct:.2f}",
        proper_pairs,
        f"{proper_pct:.2f}",
        f"{mean_depth:.2f}",
        f"{breadth_1x:.2f}",
        f"{breadth_10x:.2f}",
    ]

    with open(SUMMARY_PATH, "a", newline="") as f:
        writer = csv.writer(f, delimiter="\t")
        writer.writerow(row_data)

    # Display console output
    print(f"\nAlignment summary for {run}")
    print("----------------------------------------------")
    print(f"Total reads:             {total_reads:,}")
    print(f"Mapped reads:            {mapped_reads:,}")
    print(f"Mapping rate:            {mapping_pct:.2f}%")
    print(f"Properly paired reads:   {proper_pairs:,}")
    print(f"Properly paired rate:    {proper_pct:.2f}%")
    print(f"Mean genome depth:       {mean_depth:.2f}x")
    print(f"Genome covered >=1x:     {breadth_1x:.2f}%")
    print(f"Genome covered >=10x:    {breadth_10x:.2f}%")

# ----------------------------------------------------------
# Final Summary Display
# ----------------------------------------------------------
print("\n==============================================")
print("Alignment completed")
print(f"Finished: {time.ctime()}")
print("\nSummary Table:")
print("==============================================")

df_summary = pd.read_csv(SUMMARY_PATH, sep="\t")
print(df_summary.to_string(index=False))

LTEE Ara-3 alignment
Started: Tue Aug 11 22:57:07 2026

Reference:   data/reference/ecoli_rel606.fasta
Genome size: 4,629,812 bp
Threads:     4

Sample:     Ara3_5000
Generation: 5000
Run:        SRR2589044
Aligning reads with BWA-MEM...
Checking BAM integrity...
[OK] BAM passed samtools quickcheck.
Indexing BAM...
Generating flagstat...
Generating samtools stats...
Calculating genome-wide depth...

Alignment summary for SRR2589044
----------------------------------------------
Total reads:             1,755,042
Mapped reads:            1,753,944
Mapping rate:            99.94%
Properly paired reads:   1,681,290
Properly paired rate:    95.80%
Mean genome depth:       50.77x
Genome covered >=1x:     99.86%
Genome covered >=10x:    99.83%

Sample:     Ara3_15000
Generation: 15000
Run:        SRR2584863
Aligning reads with BWA-MEM...
Checking BAM integrity...
[OK] BAM passed samtools quickcheck.
Indexing BAM...
Generating flagstat...
Generating samtools stats...
Calculating genome-wide d

In [ ]:
from pathlib import Path
import pandas as pd

# Directories
DEPTH_DIR = Path("results/statistics/alignment")
OUT_DIR = Path("results/statistics/coverage_gaps")

OUT_DIR.mkdir(parents=True, exist_ok=True)

depth_files = sorted(DEPTH_DIR.glob("*.depth.txt"))

if not depth_files:
    raise FileNotFoundError(f"No .depth.txt files found in {DEPTH_DIR}")

for depth_path in depth_files:
    run = depth_path.name.replace(".depth.txt", "")

    print("========================================")
    print(f"Examining {run}")
    print("========================================")

    out_bed = OUT_DIR / f"{run}.zero_coverage.bed"

    # Identify zero-coverage intervals
    intervals = []
    is_open = False
    current_chr = ""
    start_pos = 0
    end_pos = 0

    with open(depth_path, "r") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 3:
                continue

            chrom, pos_str, depth_str = parts[0], int(parts[1]), int(parts[2])

            if depth_str == 0:
                if not is_open:
                    current_chr = chrom
                    start_pos = pos_str
                    is_open = True
                end_pos = pos_str
            else:
                if is_open:
                    # Convert to BED coordinates (0-based start, 1-based end)
                    intervals.append((current_chr, start_pos - 1, end_pos))
                    is_open = False

        # Handle file ending during an open zero-coverage region
        if is_open:
            intervals.append((current_chr, start_pos - 1, end_pos))

    # Write zero-coverage intervals to BED file
    with open(out_bed, "w") as f:
        for chrom, start, end in intervals:
            f.write(f"{chrom}\t{start}\t{end}\n")

    # Calculate statistics using pandas
    print("\nNumber of zero-coverage intervals:")
    num_intervals = len(intervals)
    print(num_intervals)

    if num_intervals > 0:
        df_bed = pd.DataFrame(intervals, columns=["chrom", "start", "end"])
        df_bed["length"] = df_bed["end"] - df_bed["start"]

        total_zero_bases = df_bed["length"].sum()
        print("\nTotal bases with zero coverage:")
        print(total_zero_bases)

        print("\nTen largest zero-coverage intervals:")
        df_top10 = (
            df_bed.sort_values(by="length", ascending=False)
            .head(10)
            .reset_index(drop=True)
        )
        print(df_top10.to_string(index=False))
    else:
        print("\nTotal bases with zero coverage:")
        print(0)
        print("\nTen largest zero-coverage intervals:")
        print("None found.")

    print()

Examining SRR2584863

Number of zero-coverage intervals:
6

Total bases with zero coverage:
6928

Ten largest zero-coverage intervals:
     chrom   start     end  length
CP000819.1 3895000 3901452    6452
CP000819.1 1270155 1270337     182
CP000819.1 3741966 3742141     175
CP000819.1 1270405 1270489      84
CP000819.1 1270535 1270568      33
CP000819.1 4017761 4017763       2

Examining SRR2584866

Number of zero-coverage intervals:
62

Total bases with zero coverage:
317471

Ten largest zero-coverage intervals:
     chrom   start     end  length
CP000819.1 4304956 4333227   28271
CP000819.1 2086867 2114580   27713
CP000819.1 4522345 4546463   24118
CP000819.1 4565814 4587648   21834
CP000819.1 2035094 2055107   20013
CP000819.1 4289283 4304918   15635
CP000819.1  573493  588487   14994
CP000819.1 2883967 2897416   13449
CP000819.1  592065  604413   12348
CP000819.1  558364  569028   10664

Examining SRR2589044

Number of zero-coverage intervals:
5

Total bases with zero coverage:
666

**Variant Calling**

In [ ]:
!apt-get update && apt-get install -y bcftools

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  python3-numpy python3-matplotlib texlive-latex-recommended
The following

In [ ]:
import csv
from pathlib import Path
import shutil
import subprocess
import time
import pandas as pd

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------
METADATA_PATH = Path("metadata/samples.tsv")
REF = Path("data/reference/ecoli_rel606.fasta")

BAM_DIR = Path("results/bam")
BCF_DIR = Path("results/bcf")
VCF_DIR = Path("results/vcf")
STATS_DIR = Path("results/statistics/variants")

THREADS = 4

BCF_DIR.mkdir(parents=True, exist_ok=True)
VCF_DIR.mkdir(parents=True, exist_ok=True)
STATS_DIR.mkdir(parents=True, exist_ok=True)

bcftools_bin = shutil.which("bcftools") or "bcftools"

print("==============================================")
print("LTEE Ara-3 short-variant calling")
print(f"Started: {time.ctime()}")
print("==============================================")

# ----------------------------------------------------------
# Initialize Summary Table
# ----------------------------------------------------------
SUMMARY_PATH = STATS_DIR / "variant_counts.tsv"
summary_headers = ["sample", "generation", "run", "raw_variants", "filtered_variants"]

with open(SUMMARY_PATH, "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(summary_headers)

# ----------------------------------------------------------
# Helper Functions
# ----------------------------------------------------------
def count_vcf_lines(vcf_path, extra_args=None):
    """Counts headerless records (-H) from a VCF using bcftools view."""
    cmd = [bcftools_bin, "view", "-H", str(vcf_path)]
    if extra_args:
        cmd[2:2] = extra_args  # Insert extra filter flags before -H

    res = subprocess.run(cmd, capture_output=True, text=True, check=True)
    lines = res.stdout.strip().split("\n")
    return len(lines) if lines != [""] else 0

# ----------------------------------------------------------
# Process Samples
# ----------------------------------------------------------


for _, row in df_samples.iterrows():

    bam = BAM_DIR / f"{run}.aligned.sorted.bam"
    raw_bcf = BCF_DIR / f"{run}.raw.bcf"
    raw_vcf = VCF_DIR / f"{run}.raw.vcf.gz"
    filtered_vcf = VCF_DIR / f"{run}.filtered.vcf.gz"

    if not bam.exists() or bam.stat().st_size == 0:
        raise FileNotFoundError(f"BAM not found: {bam}")

    # 1. Generate genotype likelihoods (bcftools mpileup | bcftools view)
    if raw_bcf.exists() and raw_bcf.stat().st_size > 0:
        print(f"[SKIP] Existing BCF: {raw_bcf}")
    else:
        print("Running bcftools mpileup...")
        cmd_mpileup = [
            bcftools_bin,
            "mpileup",
            "--threads",
            str(THREADS),
            "-Ou",
            "-f",
            str(REF),
            "-a",
            "FORMAT/AD,FORMAT/DP",
            str(bam),
        ]
        cmd_view = [
            bcftools_bin,
            "view",
            "-Ob",
            "-o",
            str(raw_bcf),
        ]

        p1 = subprocess.Popen(cmd_mpileup, stdout=subprocess.PIPE)
        p2 = subprocess.Popen(cmd_view, stdin=p1.stdout, stdout=subprocess.PIPE)
        p1.stdout.close()
        p2.communicate()

        if p2.returncode != 0:
            raise RuntimeError(f"mpileup generation failed for {run}")

    # 2. Haploid variant calling
    if raw_vcf.exists() and raw_vcf.stat().st_size > 0:
        print(f"[SKIP] Existing raw VCF: {raw_vcf}")
    else:
        print("Calling haploid variants...")
        cmd_call = [
            bcftools_bin,
            "call",
            "--threads",
            str(THREADS),
            "--ploidy",
            "1",
            "-m",
            "-v",
            "-Oz",
            "-o",
            str(raw_vcf),
            str(raw_bcf),
        ]
        subprocess.run(cmd_call, check=True)
        subprocess.run([bcftools_bin, "index", "-t", str(raw_vcf)], check=True)

    # 3. Quality filtering
    if filtered_vcf.exists() and filtered_vcf.stat().st_size > 0:
        print(f"[SKIP] Existing filtered VCF: {filtered_vcf}")
    else:
        print("Filtering variants...")
        cmd_filter = [
            bcftools_bin,
            "filter",
            "--threads",
            str(THREADS),
            "-e",
            "QUAL<20 || FORMAT/DP<10",
            "-s",
            "LowQual",
            "-Oz",
            "-o",
            str(filtered_vcf),
            str(raw_vcf),
        ]
        subprocess.run(cmd_filter, check=True)
        subprocess.run([bcftools_bin, "index", "-t", str(filtered_vcf)], check=True)

    # 4. Count variants
    raw_count = count_vcf_lines(raw_vcf)
    pass_count = count_vcf_lines(filtered_vcf, extra_args=["-f", "PASS"])

    # 5. bcftools stats
    stats_out = STATS_DIR / f"{run}.bcftools_stats.txt"
    with open(stats_out, "w") as out:
        subprocess.run([bcftools_bin, "stats", str(filtered_vcf)], stdout=out, check=True)

    # 6. Variant type counts
    snp_count = count_vcf_lines(filtered_vcf, extra_args=["-f", "PASS", "-v", "snps"])
    indel_count = count_vcf_lines(filtered_vcf, extra_args=["-f", "PASS", "-v", "indels"])

    print(f"\nRaw variants:      {raw_count}")
    print(f"PASS variants:     {pass_count}")
    print(f"PASS SNPs:         {snp_count}")
    print(f"PASS indels:       {indel_count}")

    # Append summary row
    with open(SUMMARY_PATH, "a", newline="") as f:
        writer = csv.writer(f, delimiter="\t")
        writer.writerow([sample, generation, run, raw_count, pass_count])

# ----------------------------------------------------------
# Final Output
# ----------------------------------------------------------
print("\n==============================================")
print("Variant calling completed")
print(f"Finished: {time.ctime()}")
print("==============================================")

df_summary = pd.read_csv(SUMMARY_PATH, sep="\t")
print(df_summary.to_string(index=False))

LTEE Ara-3 short-variant calling
Started: Tue Aug 11 23:11:02 2026
Running bcftools mpileup...
Calling haploid variants...
Filtering variants...

Raw variants:      13
PASS variants:     9
PASS SNPs:         6
PASS indels:       3
[SKIP] Existing BCF: results/bcf/SRR2589044.raw.bcf
[SKIP] Existing raw VCF: results/vcf/SRR2589044.raw.vcf.gz
[SKIP] Existing filtered VCF: results/vcf/SRR2589044.filtered.vcf.gz

Raw variants:      13
PASS variants:     9
PASS SNPs:         6
PASS indels:       3
[SKIP] Existing BCF: results/bcf/SRR2589044.raw.bcf
[SKIP] Existing raw VCF: results/vcf/SRR2589044.raw.vcf.gz
[SKIP] Existing filtered VCF: results/vcf/SRR2589044.filtered.vcf.gz

Raw variants:      13
PASS variants:     9
PASS SNPs:         6
PASS indels:       3

Variant calling completed
Finished: Tue Aug 11 23:12:08 2026
    sample  generation        run  raw_variants  filtered_variants
Ara3_50000       50000 SRR2589044            13                  9
Ara3_50000       50000 SRR2589044        

**PASS Variant Extraction, Normalization, and Summary**

In [ ]:
!pip install pysam

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 51.2 MB/s eta 0:00:00


In [ ]:
import csv
from pathlib import Path
import shutil
import subprocess
import time
import pandas as pd
import pysam

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------
REF_FASTA = Path("data/reference/ecoli_rel606.fasta")
VCF_DIR = Path("results/vcf")
STATS_DIR = Path("results/statistics/variants")

SUMMARY_PATH = STATS_DIR / "pass_variant_summary.tsv"

STATS_DIR.mkdir(parents=True, exist_ok=True)

bcftools_bin = shutil.which("bcftools") or "bcftools"

# ----------------------------------------------------------
# Helper Functions
# ----------------------------------------------------------
def parse_tstv_from_stats(stats_file_path):
    """Extracts the Ts/Tv ratio from a bcftools stats text file."""
    if not stats_file_path.exists():
        return "NA"

    with open(stats_file_path, "r") as f:
        for line in f:
            fields = line.strip().split("\t")
            if fields[0] == "TSTV" and len(fields) >= 5:
                return fields[4]
    return "NA"


def get_vcf_variant_counts(vcf_path):
    """Counts total records, SNPs, and Indels using pysam."""
    vcf = pysam.VariantFile(str(vcf_path))

    total = 0
    snps = 0
    indels = 0

    for record in vcf:
        total += 1
        for alt in record.alts:
            if len(record.ref) == 1 and len(alt) == 1:
                snps += 1
            else:
                indels += 1

    vcf.close()
    return total, snps, indels


# ----------------------------------------------------------
# Initialize Summary File
# ----------------------------------------------------------
summary_headers = [
    "sample",
    "generation",
    "run",
    "pass_variants",
    "snps",
    "indels",
    "ts_tv",
]

with open(SUMMARY_PATH, "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(summary_headers)

# ----------------------------------------------------------
# Process Samples
# ----------------------------------------------------------
df_samples = pd.read_csv(METADATA_PATH, sep="\t")

for _, row in df_samples.iterrows():

    print("==============================================")
    print(f"Preparing PASS variants for {run}")
    print("==============================================")

    input_vcf = VCF_DIR / f"{run}.filtered.vcf.gz"
    pass_vcf = VCF_DIR / f"{run}.PASS.vcf.gz"
    norm_vcf = VCF_DIR / f"{run}.PASS.norm.vcf.gz"
    stats_file = STATS_DIR / f"{run}.PASS.bcftools_stats.txt"

    # 1. Extract PASS records only
    subprocess.run(
        [
            bcftools_bin,
            "view",
            "-f",
            "PASS",
            "-Oz",
            "-o",
            str(pass_vcf),
            str(input_vcf),
        ],
        check=True,
    )
    pysam.tabix_index(str(pass_vcf), preset="vcf", force=True)

    # 2. Normalize representation
    subprocess.run(
        [
            bcftools_bin,
            "norm",
            "-f",
            str(REF_FASTA),
            "-m",
            "-any",
            "-Oz",
            "-o",
            str(norm_vcf),
            str(pass_vcf),
        ],
        check=True,
    )
    pysam.tabix_index(str(norm_vcf), preset="vcf", force=True)

    # 3. Generate bcftools stats
    with open(stats_file, "w") as out:
        subprocess.run(
            [bcftools_bin, "stats", str(norm_vcf)],
            stdout=out,
            check=True,
        )

    # 4. Extract metrics
    total, snps, indels = get_vcf_variant_counts(norm_vcf)
    tstv = parse_tstv_from_stats(stats_file)

    # 5. Append row to summary
    with open(SUMMARY_PATH, "a", newline="") as f:
        writer = csv.writer(f, delimiter="\t")
        writer.writerow([sample, generation, run, total, snps, indels, tstv])

# ----------------------------------------------------------
# Final Summary Output
# ----------------------------------------------------------
print()
df_summary = pd.read_csv(SUMMARY_PATH, sep="\t")
print(df_summary.to_string(index=False))

Preparing PASS variants for SRR2589044
Preparing PASS variants for SRR2589044
Preparing PASS variants for SRR2589044

    sample  generation        run  pass_variants  snps  indels  ts_tv
Ara3_50000       50000 SRR2589044              9     6       3    2.0
Ara3_50000       50000 SRR2589044              9     6       3    2.0
Ara3_50000       50000 SRR2589044              9     6       3    2.0


**Mutation Spectrum Analysis**

In [ ]:
import csv
from pathlib import Path
import pandas as pd
import pysam

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------

VCF_DIR = Path("results/vcf")
OUT_DIR = Path("results/statistics/mutation_spectrum")

OUT_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = OUT_DIR / "substitution_spectrum.tsv"

# Reverse complement map for pyrimidine-centric collapsing
COMPLEMENT = {"A": "T", "T": "A", "C": "G", "G": "C"}

CANONICAL_TYPES = ["A>G", "A>C", "A>T", "C>T", "C>G", "C>A"]


def collapse_substitution(ref, alt):
    """Collapses purine bases (T, G) to pyrimidine equivalents (A, C)."""
    ref, alt = ref.upper(), alt.upper()

    if ref in ("T", "G"):
        ref = COMPLEMENT[ref]
        alt = COMPLEMENT[alt]

    return f"{ref}>{alt}"


# ----------------------------------------------------------
# Initialize Summary File
# ----------------------------------------------------------
summary_headers = ["sample", "generation", "run", "substitution", "count"]

with open(SUMMARY_PATH, "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(summary_headers)

# ----------------------------------------------------------
# Process Samples
# ----------------------------------------------------------


for _, row in df_samples.iterrows():

    vcf_path = VCF_DIR / f"{run}.PASS.norm.vcf.gz"

    if not vcf_path.exists():
        print(f"[WARNING] Skipping missing VCF: {vcf_path}")
        continue

    print(f"Processing {run}...")

    # Initialize count spectrum
    spectrum_counts = {sub_type: 0 for sub_type in CANONICAL_TYPES}

    vcf = pysam.VariantFile(str(vcf_path))

    for record in vcf:
        # Evaluate single-nucleotide variants
        if len(record.ref) == 1:
            for alt in record.alts:
                if len(alt) == 1 and alt != record.ref:
                    sub_type = collapse_substitution(record.ref, alt)
                    if sub_type in spectrum_counts:
                        spectrum_counts[sub_type] += 1

    vcf.close()

    # Append results to TSV summary
    with open(SUMMARY_PATH, "a", newline="") as f:
        writer = csv.writer(f, delimiter="\t")
        for sub_type in CANONICAL_TYPES:
            writer.writerow(
                [sample, generation, run, sub_type, spectrum_counts[sub_type]]
            )

# ----------------------------------------------------------
# Final Summary Output
# ----------------------------------------------------------
print()
df_summary = pd.read_csv(SUMMARY_PATH, sep="\t")
print(df_summary.to_string(index=False))

Processing SRR2589044...
Processing SRR2589044...
Processing SRR2589044...

    sample  generation        run substitution  count
Ara3_50000       50000 SRR2589044          A>G      2
Ara3_50000       50000 SRR2589044          A>C      0
Ara3_50000       50000 SRR2589044          A>T      0
Ara3_50000       50000 SRR2589044          C>T      2
Ara3_50000       50000 SRR2589044          C>G      0
Ara3_50000       50000 SRR2589044          C>A      2
Ara3_50000       50000 SRR2589044          A>G      2
Ara3_50000       50000 SRR2589044          A>C      0
Ara3_50000       50000 SRR2589044          A>T      0
Ara3_50000       50000 SRR2589044          C>T      2
Ara3_50000       50000 SRR2589044          C>G      0
Ara3_50000       50000 SRR2589044          C>A      2
Ara3_50000       50000 SRR2589044          A>G      2
Ara3_50000       50000 SRR2589044          A>C      0
Ara3_50000       50000 SRR2589044          A>T      0
Ara3_50000       50000 SRR2589044          C>T      2
Ara3_5

**SnpEff Variant Annotation**
It runs snpEff via a piped subprocess stream directly into bgzip (avoiding uncompressed disk writes), indexes the resulting compressed VCF using pysam, and parses the SnpEff CSV stats files into a consolidated summary DataFrame.

In [ ]:
!apt-get install -y default-jre -qq

!curl -L -o snpEff_latest_core.zip https://snpeff-public.s3.amazonaws.com/versions/snpEff_latest_core.zip
!file snpEff_latest_core.zip
!unzip -q snpEff_latest_core.zip
!java -jar snpEff/snpEff.jar -version

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 63.5M  100 63.5M    0     0  21.0M      0  0:00:03  0:00:03 --:--:-- 21.0M
snpEff_latest_core.zip: Zip archive data, at least v1.0 to extract, compression method=store
[0.005s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.005s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SnpEff	5.4c	2026-02-23


In [ ]:
!mkdir -p /opt
!ln -sf $(pwd)/snpEff /opt/snpEff
!snpEff -version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SnpEff	5.4c	2026-02-23


In [ ]:
!snpEff -version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SnpEff	5.4c	2026-02-23


In [ ]:
!apt-get install -y tabix
!which bgzip
!bgzip --version

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tabix
0 upgraded, 1 newly installed, 0 to remove and 23 not upgraded.
Need to get 351 kB of archives.
After this operation, 1,179 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tabix amd64 1.13+ds-2build1 [351 kB]
Fetched 351 kB in 1s (456 kB/s)
Selecting previously unselected package tabix.
(Reading database ... 119569 files and directories currently installed.)
Preparing to unpack .../tabix_1.13+ds-2build1_amd64.deb ...
Unpacking tabix (1.13+ds-2build1) ...
Setting up tabix (1.13+ds-2build1) ...
Processing triggers for man-db (2.10.2-1) ...
/usr/bin/bgzip
bgzip (htslib) 1.13+ds
Copyright (C) 2021 Genome Research Ltd.


In [ ]:
!java -jar snpEff/snpEff.jar GRCh38.86 input.vcf > results/annotation/SRR2589044.annotated.vcf 2> results/annotation/snpeff.log

In [ ]:
# Build the database using the local files
!java -Xlog:disable -jar ${SNPEFF_DIR}/snpEff.jar build -c ${CONFIG_FILE} -gff3 -v REL606

Error: Unable to access jarfile /snpEff.jar


In [ ]:
# Clean old files
!rm -f results/annotation/SRR2589044.annotated.vcf*

# Run annotation cleanly (stderr suppressed so JVM warnings don't pollute VCF stream)
!java -Xlog:disable -jar /content/snpEff/snpEff.jar eff \
  -c /content/snpEff/snpEff.config \
  -v REL606 \
  results/vcf/SRR2589044.PASS.norm.vcf.gz 2>/dev/null \
  | bgzip -c > results/annotation/SRR2589044.annotated.vcf.gz

# Index the output
!tabix -p vcf results/annotation/SRR2589044.annotated.vcf.gz

In [ ]:
!java -jar snpEff/snpEff.jar databases | grep REL606

REL606                                                      	Escherichia_coli_B_str_REL606                               	          	                              	[https://snpeff-public.s3.amazonaws.com/databases/v5_4/snpEff_v5_4_REL606.zip, https://snpeff-public.s3.amazonaws.com/databases/v5_3/snpEff_v5_3_REL606.zip, https://snpeff-public.s3.amazonaws.com/databases/v5_2/snpEff_v5_2_REL606.zip, https://snpeff-public.s3.amazonaws.com/databases/v5_1/snpEff_v5_1_REL606.zip, https://snpeff-public.s3.amazonaws.com/databases/v5_0/snpEff_v5_0_REL606.zip]


In [ ]:
!ls -la snpEff/ | grep -i config
!ls -la snpeff/ 2>&1   # likely "No such file or directory" or a different unrelated folder

-rw-r--r-- 1 root root 19152562 Aug 12 00:00 snpEff.config
ls: cannot access 'snpeff/': No such file or directory


In [ ]:
import os
p = "results/annotation/SRR2589044.annotated.vcf.gz"
if os.path.exists(p):
    os.remove(p)

In [ ]:
!head -20 snpEff/snpEff.config
!wc -l snpEff/snpEff.config
!tail -20 snpEff/snpEff.config


#---
# Core 
#---

#-------------------------------------------------------------------------------
#
# SnpEff configuration file
#
#																Pablo Cingolani
#-------------------------------------------------------------------------------

#---
# Databases are stored here
# E.g.: Information for 'hg19' is stored in data.dir/hg19/
#
# You can use tilde ('~') as first character to refer to your home directory. 
# Also, a non-absolute path will be relative to config's file dir
# 
#---
319418 snpEff/snpEff.config
_propionibacterium_namnetense_sk182b_jcvi.retrieval_date : 2020-01-26

_pseudomonas_geniculata_atcc_19374_jcm_13324.genome : _pseudomonas_geniculata_atcc_19374_jcm_13324
_pseudomonas_geniculata_atcc_19374_jcm_13324.reference : ftp.ensemblgenomes.org/pub/release-46
_pseudomonas_geniculata_atcc_19374_jcm_13324.retrieval_date : 2020-01-26

_pseudomonas_syringae_pv_tomato_str_dc3000.genome : _pseudomonas_syringae_pv_tomato_str_dc3000
_pseudomonas_syringae_pv_tomato_str_dc3000.

In [ ]:
with open("snpEff/snpEff.config", "a") as f:
    f.write("\n# LTEE Ara-3 ancestor\nREL606.genome : Escherichia coli B REL606\n")

In [ ]:
!mkdir -p snpEff/data/REL606
!curl -L -o snpEff/data/REL606/genes.gbk "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=CP000819.1&rettype=gbwithparts&retmode=text"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 9698k    0 9698k    0     0  8284k      0 --:--:--  0:00:01 --:--:-- 8289k


In [ ]:
!head -20 snpEff/data/REL606/genes.gbk
!grep -c "^LOCUS" snpEff/data/REL606/genes.gbk

LOCUS       CP000819             4629812 bp    DNA     circular BCT 31-JAN-2014
DEFINITION  Escherichia coli B str. REL606, complete genome.
ACCESSION   CP000819
VERSION     CP000819.1
DBLINK      BioProject: PRJNA18281
            BioSample: SAMN02603421
KEYWORDS    .
SOURCE      Escherichia coli B str. REL606
  ORGANISM  Escherichia coli B str. REL606
            Bacteria; Pseudomonadati; Pseudomonadota; Gammaproteobacteria;
            Enterobacterales; Enterobacteriaceae; Escherichia.
REFERENCE   1  (bases 1 to 4629812)
  AUTHORS   Jeong,H., Barbe,V., Vallenet,D., Choi,S.-H., Lee,C.H., Lee,S.-W.,
            Vacherie,B., Yoon,S.H., Yu,D.-S., Cattolico,L., Hur,C.-G.,
            Park,H.-S., Segurens,B., Blot,M., Schneider,D., Studier,F.W.,
            Oh,T.K., Lenski,R.E., Daegelen,P. and Kim,J.F.
  CONSRTM   International E. coli B Consortium
  TITLE     Complete genome sequence of Escherichia coli (B) REL606
  JOURNAL   Unpublished
REFERENCE   2  (bases 1 to 4629812)
1


In [ ]:
!java -jar snpEff/snpEff.jar build -genbank -v REL606 -c snpEff/snpEff.config

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
00:00:00 SnpEff version SnpEff 5.4c (build 2026-02-23 07:25), by Pablo Cingolani
00:00:00 Command: 'build'
00:00:00 Building database for 'REL606'
00:00:00 Reading configuration file 'snpEff/snpEff.config'. Genome: 'REL606'
00:00:00 Looking for config file: '/content/snpEff/snpEff.config'
00:00:00 Reading config file: /content/snpEff/snpEff.config
00:00:02 done
00:00:04 Chromosome: 'CP000819.1'	length: 4629812
00:00:04 Create exons from CDS (if needed): 
....................................................................................................................................................................................................................

In [ ]:
!java -jar snpEff/snpEff.jar databases | grep -i rel606 2>/dev/null
!ls snpEff/data/REL606/

Escherichia_coli_b_str_rel606                               	Escherichia_coli_b_str_rel606                               	          	                              	[https://snpeff-public.s3.amazonaws.com/databases/v5_4/snpEff_v5_4_Escherichia_coli_b_str_rel606.zip, https://snpeff-public.s3.amazonaws.com/databases/v5_3/snpEff_v5_3_Escherichia_coli_b_str_rel606.zip, https://snpeff-public.s3.amazonaws.com/databases/v5_2/snpEff_v5_2_Escherichia_coli_b_str_rel606.zip, https://snpeff-public.s3.amazonaws.com/databases/v5_1/snpEff_v5_1_Escherichia_coli_b_str_rel606.zip, https://snpeff-public.s3.amazonaws.com/databases/v5_0/snpEff_v5_0_Escherichia_coli_b_str_rel606.zip]
REL606                                                      	Escherichia coli B REL606                                   	OK        	                              	[https://snpeff-public.s3.amazonaws.com/databases/v5_4/snpEff_v5_4_REL606.zip, https://snpeff-public.s3.amazonaws.com/databases/v5_3/snpEff_v5_3_REL606.zip, https://s

In [ ]:
# 1. Confirm file structure - header and first records
!zcat results/annotation/SRR2589044.annotated.vcf.gz | head -30

# 2. Check chromosome/contig order - this is the most likely culprit
!zcat results/annotation/SRR2589044.annotated.vcf.gz | grep -v "^#" | cut -f1 | uniq -c

# 3. Check position ordering within each contig
!zcat results/annotation/SRR2589044.annotated.vcf.gz | grep -v "^#" | awk '{print $1, $2}' | head -30

# 4. Check total record count and look at the tail
!zcat results/annotation/SRR2589044.annotated.vcf.gz | grep -vc "^#"
!zcat results/annotation/SRR2589044.annotated.vcf.gz | tail -10

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed">
##bcftoolsVersion=1.13+htslib-1.13+ds
##bcftoolsCommand=mpileup --threads 4 -Ou -f data/reference/ecoli_rel606.fasta -a FORMAT/AD,FORMAT/DP results/bam/SRR2589044.aligned.sorted.bam
##reference=file://data/reference/ecoli_rel606.fasta
##contig=<ID=REL606,length=4629812>
##ALT=<ID=*,Description="Represents allele(s) other than observed.">
##INFO=<ID=INDEL,Number=0,Type=Flag,Description="Indicates that the variant is an INDEL.">
##INFO=<ID=IDV,Number=1,Type=Integer,Description="Maximum number of raw reads supporting an indel">
##INFO=<ID=IMF,Number=1,Type=Float,Description="Maximum fraction of

In [ ]:
!tabix -p vcf results/annotation/SRR2589044.annotated.vcf.gz

[E::get_intv] Failed to parse TBX_VCF, was wrong -p [type] used?
The offending line was: "[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate"
[E::get_intv] Failed to parse TBX_VCF, was wrong -p [type] used?
The offending line was: "[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate"


In [ ]:
# 1. Confirm the file doesn't have the JVM warning contamination anymore
!zcat results/annotation/SRR2589044.annotated.vcf.gz | head -3

# 2. Confirm whether annotations are still failing
!zcat results/annotation/SRR2589044.annotated.vcf.gz | grep -c "ERROR_CHROMOSOME_NOT_FOUND"
!zcat results/annotation/SRR2589044.annotated.vcf.gz | grep -v "^#" | head -3

# 3. Check what chromosome name your REL606 database actually expects
!java -jar snpEff/snpEff.jar dump REL606 -c snpEff/snpEff.config 2>/dev/null | grep -i "chromosome\|contig" | head -5
!grep "^LOCUS" snpEff/data/REL606/genes.gbk

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
##fileformat=VCFv4.2
9
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
REL606	648692	.	C	T	225.417	PASS	DP=44;VDB=0.21047;SGB=-0.693146;FS=0;MQ0F=0;AC=1;AN=1;DP4=0,0,32,11;MQ=60;ANN=T||MODIFIER|||||||||||||ERROR_CHROMOSOME_NOT_FOUND	GT:PL:DP:AD	1:255,0:43:0,43
# Number of chromosomes      : 1
# Chromosomes                : Format 'chromo_name size codon_table'
LOCUS       CP000819       

In [ ]:
import os
p = "results/annotation/SRR2589044.annotated.vcf.gz"
tbi = p + ".tbi"
for f in (p, tbi):
    if os.path.exists(f):
        os.remove(f)

In [ ]:
import csv
from pathlib import Path
import shutil
import subprocess
import time
import os
import pandas as pd
import pysam

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------
METADATA_PATH = Path("metadata/samples.tsv")
VCF_DIR = Path("results/vcf")
OUT_DIR = Path("results/annotation")
STATS_DIR = Path("results/statistics/annotation")

SNPEFF_CONFIG = Path("snpEff/snpEff.config")   # capital E
SNPEFF_JAR = Path("snpEff/snpEff.jar")         # capital E  # <-- point directly at your working jar, not the PATH wrapper
GENOME = "REL606"
CHR_MAP_PATH = Path("chr_map.txt")

# Create chromosome mapping file (CP000819.1 -> REL606)
CHR_MAP_PATH.write_text("CP000819.1\tCP000819\n")

OUT_DIR.mkdir(parents=True, exist_ok=True)
STATS_DIR.mkdir(parents=True, exist_ok=True)

# Binary discovery (snpEff now goes through the jar directly, so it's excluded here)
bcftools_bin = shutil.which("bcftools") or "bcftools"
bgzip_bin = shutil.which("bgzip") or "bgzip"

# Sanity check the jar actually exists before running anything
if not SNPEFF_JAR.exists():
    raise FileNotFoundError(
        f"snpEff jar not found at {SNPEFF_JAR} — update SNPEFF_JAR to your actual install path"
    )

print("==============================================")
print("LTEE Ara-3 variant annotation")
print(f"Started: {time.ctime()}")
print("==============================================")

# Load metadata
df_samples = pd.read_csv(METADATA_PATH, sep="\t")
df_samples.columns = df_samples.columns.str.upper()

# ----------------------------------------------------------
# Process Samples
# ----------------------------------------------------------
for _, row in df_samples.iterrows():
    sample = str(row["SAMPLE"])
    generation = str(row["GENERATION"])
    run = str(row["RUN"])

    print("\n==============================================")
    print(f"Sample:     {sample}")
    print(f"Generation: {generation}")
    print(f"Run:        {run}")
    print("==============================================")

    input_vcf = VCF_DIR / f"{run}.PASS.norm.vcf.gz"
    output_vcf = OUT_DIR / f"{run}.annotated.vcf.gz"
    html_stats = STATS_DIR / f"{run}.snpeff_summary.html"
    csv_stats = STATS_DIR / f"{run}.snpeff_stats.csv"

    if not input_vcf.exists() or input_vcf.stat().st_size == 0:
        print(f"[SKIP] Missing input VCF: {input_vcf}")
        continue

    # Skip only if output exists, is non-empty, AND has a valid tabix index
    # (a leftover broken file will fail the index check and get regenerated)
    if output_vcf.exists() and output_vcf.stat().st_size > 0:
        tbi_path = Path(str(output_vcf) + ".tbi")
        if tbi_path.exists():
            print(f"[SKIP] Annotation already exists and is indexed:\n{output_vcf}")
            continue
        else:
            print(f"[REBUILD] Existing output has no valid index, regenerating: {output_vcf}")
            os.remove(output_vcf)

    print("Annotating variants (renaming CP000819.1 -> REL606)...")

    # Command 1: Rename chromosome with bcftools
    cmd_rename = [
        bcftools_bin,
        "annotate",
        "--rename-chrs",
        str(CHR_MAP_PATH),
        str(input_vcf),
    ]

    # Command 2: Annotate with snpEff via jar directly (reads stdin '-')
    cmd_snpeff = [
    "java", "-Xlog:all=off", "-jar", str(SNPEFF_JAR),
    "-c", str(SNPEFF_CONFIG),
    "-v",
    "-stats", str(html_stats),
    "-csvStats", str(csv_stats),
    GENOME,
    "-",
    ]

    # Pipeline stream: bcftools annotate | snpEff - | bgzip -c > output_vcf
    with open(output_vcf, "wb") as out_f:
        p_rename = subprocess.Popen(cmd_rename, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        p_snpeff = subprocess.Popen(
            cmd_snpeff, stdin=p_rename.stdout, stdout=subprocess.PIPE, stderr=subprocess.PIPE
        )
        p_bgzip = subprocess.Popen(
            [bgzip_bin, "-c"], stdin=p_snpeff.stdout, stdout=out_f
        )

        # Allow p_rename and p_snpeff to receive SIGPIPE if downstream exits
        p_rename.stdout.close()
        p_snpeff.stdout.close()

        p_bgzip.communicate()
        _, snpeff_stderr = p_snpeff.communicate()
        _, rename_stderr = p_rename.communicate()

        p_rename.wait()
        p_snpeff.wait()

        if p_rename.returncode != 0:
            raise RuntimeError(
                f"bcftools annotate failed for {run}: exit {p_rename.returncode}\n"
                f"{rename_stderr.decode(errors='replace')}"
            )
        if p_snpeff.returncode != 0:
            raise RuntimeError(
                f"snpEff failed for {run}: exit {p_snpeff.returncode}\n"
                f"{snpeff_stderr.decode(errors='replace')}"
            )
        if p_bgzip.returncode != 0:
            raise RuntimeError(f"bgzip failed for {run}")

    # Sanity check output actually looks like VCF before indexing
    with open(output_vcf, "rb") as f:
        magic = f.read(2)
    if magic != b"\x1f\x8b":  # gzip/bgzip magic bytes
        raise RuntimeError(f"Output for {run} is not a valid gzip/bgzip file")

    # Index annotated output VCF
    pysam.tabix_index(str(output_vcf), preset="vcf", force=True)

    # Final validation: confirm pysam can actually open and fetch a record
    try:
        vcf_check = pysam.VariantFile(str(output_vcf))
        next(vcf_check.fetch())
        vcf_check.close()
    except StopIteration:
        print(f"[WARN] {run}: indexed OK but contains zero records")
    except Exception as e:
        raise RuntimeError(f"Post-annotation validation failed for {run}: {e}")

    print(f"[OK] Annotated {run}")

# ----------------------------------------------------------
# Final Summary Output
# ----------------------------------------------------------
print("\n==============================================")
print("Annotation complete")
print(f"Finished: {time.ctime()}")
print("==============================================")

LTEE Ara-3 variant annotation
Started: Wed Aug 12 00:26:23 2026

Sample:     Ara3_5000
Generation: 5000
Run:        SRR2589044
Annotating variants (renaming CP000819.1 -> REL606)...
[OK] Annotated SRR2589044

Sample:     Ara3_15000
Generation: 15000
Run:        SRR2584863
[SKIP] Missing input VCF: results/vcf/SRR2584863.PASS.norm.vcf.gz

Sample:     Ara3_50000
Generation: 50000
Run:        SRR2584866
[SKIP] Missing input VCF: results/vcf/SRR2584866.PASS.norm.vcf.gz

Annotation complete
Finished: Wed Aug 12 00:26:31 2026


**Annotation Extraction Script**

In [ ]:
# Test file integrity
!bcftools view results/annotation/SRR2589044.annotated.vcf.gz | head

Failed to read from results/annotation/SRR2589044.annotated.vcf.gz: unknown file type


In [ ]:
!head -n 5 results/annotation/SRR2589044.annotated.vcf.gz

�     � BC � ��1� ��S��F!��cdR �	�̲$��;�	��_���Nӽ/�ᥦ�.3�k�>U��Ux4*,'WΙ���0��ݽ�_iЉJ�26�
���uֺ�h���r��HJA)"�����jO�a�W�̿-C+~ ��2�I  �     � BC           

In [ ]:
import csv
import gzip
from pathlib import Path
import pandas as pd

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------
METADATA_PATH = Path("metadata/samples.tsv")
ANN_DIR = Path("results/annotation")
OUT_DIR = Path("results/statistics/annotation")

OUT_DIR.mkdir(parents=True, exist_ok=True)

ALL_SUMMARY_PATH = OUT_DIR / "all_annotated_variants.tsv"

HEADERS = [
    "sample",
    "generation",
    "run",
    "chrom",
    "position",
    "ref",
    "alt",
    "qual",
    "allele",
    "consequence",
    "impact",
    "gene",
    "gene_id",
    "feature_type",
    "feature_id",
    "biotype",
    "hgvs_c",
    "hgvs_p",
]

# Write header to combined summary
with open(ALL_SUMMARY_PATH, "w", newline="") as f:
    writer = csv.writer(f, delimiter="\t")
    writer.writerow(HEADERS)

# Read metadata and standardize column names
df_samples = pd.read_csv(METADATA_PATH, sep="\t")
df_samples.columns = df_samples.columns.str.upper()

# ----------------------------------------------------------
# Helper Functions
# ----------------------------------------------------------
def open_vcf(path):
    """Opens both bgzip/gzip and plain text VCF files safely."""
    try:
        f = gzip.open(path, "rt")
        f.read(1)
        f.seek(0)
        return f
    except (gzip.BadGzipFile, OSError):
        return open(path, "r")


# ----------------------------------------------------------
# Process Samples
# ----------------------------------------------------------
total_annotation_records = 0

for _, row in df_samples.iterrows():
    sample = str(row["SAMPLE"])
    generation = str(row["GENERATION"])
    run = str(row["RUN"])

    print("==============================================")
    print(f"Extracting annotations: {run}")
    print("==============================================")

    # Dynamic file resolution using current 'run'
    vcf_path = ANN_DIR / f"{run}.annotated.vcf.gz"
    sample_out_path = OUT_DIR / f"{run}.annotated_variants.tsv"

    if not vcf_path.exists() or vcf_path.stat().st_size == 0:
        print(f"[WARNING] Skipping missing or empty VCF: {vcf_path}")
        continue

    sample_rows = []

    with open_vcf(vcf_path) as vcf_file:
        for line in vcf_file:
            if line.startswith("#"):
                continue

            cols = line.strip().split("\t")
            if len(cols) < 8:
                continue

            chrom = cols[0]
            pos = cols[1]
            ref = cols[3]
            alt = cols[4]
            qual = cols[5]
            info = cols[7]

            # Extract SnpEff field (supports both ANN and EFF)
            ann_str = None
            for item in info.split(";"):
                if item.startswith("ANN="):
                    ann_str = item[4:]
                    break
                elif item.startswith("EFF="):
                    ann_str = item[4:]
                    break

            if ann_str:
                for entry in ann_str.split(","):
                    fields = entry.split("|")

                    # Pad fields list safely
                    while len(fields) < 11:
                        fields.append("")

                    allele = fields[0]
                    consequence = fields[1]
                    impact = fields[2]
                    gene = fields[3]
                    gene_id = fields[4]
                    feature_type = fields[5]
                    feature_id = fields[6]
                    biotype = fields[7]
                    hgvs_c = fields[9]
                    hgvs_p = fields[10]

                    sample_rows.append(
                        [
                            sample,
                            generation,
                            run,
                            chrom,
                            pos,
                            ref,
                            alt,
                            qual,
                            allele,
                            consequence,
                            impact,
                            gene,
                            gene_id,
                            feature_type,
                            feature_id,
                            biotype,
                            hgvs_c,
                            hgvs_p,
                        ]
                    )

    # Write sample output TSV
    with open(sample_out_path, "w", newline="") as f:
        writer = csv.writer(f, delimiter="\t")
        writer.writerow(HEADERS)
        writer.writerows(sample_rows)

    # Append to combined table
    with open(ALL_SUMMARY_PATH, "a", newline="") as f:
        writer = csv.writer(f, delimiter="\t")
        writer.writerows(sample_rows)

    total_annotation_records += len(sample_rows)
    print(f"[OK] {sample_out_path} (Extracted {len(sample_rows)} records)")

# ----------------------------------------------------------
# Summary Output
# ----------------------------------------------------------
print("\n==============================================")
print("Annotation extraction complete")
print("==============================================")
print(f"Combined table: {ALL_SUMMARY_PATH}")
print(f"Total annotation records extracted: {total_annotation_records}")

Extracting annotations: SRR2589044
[OK] results/statistics/annotation/SRR2589044.annotated_variants.tsv (Extracted 0 records)
Extracting annotations: SRR2584863
[WARNING] Skipping missing or empty VCF: results/annotation/SRR2584863.annotated.vcf.gz
Extracting annotations: SRR2584866
[WARNING] Skipping missing or empty VCF: results/annotation/SRR2584866.annotated.vcf.gz

Annotation extraction complete
Combined table: results/statistics/annotation/all_annotated_variants.tsv
Total annotation records extracted: 0


In [ ]:
import gzip
from pathlib import Path
import pandas as pd

METADATA_PATH = Path("metadata/samples.tsv")
ANN_DIR = Path("results/annotation")
VCF_DIR = Path("results/vcf")

df_samples = pd.read_csv(METADATA_PATH, sep="\t")
df_samples.columns = df_samples.columns.str.upper()

print("File Status Check:")
print("=" * 60)

for _, row in df_samples.iterrows():
    run = str(row["RUN"])

    pass_vcf = VCF_DIR / f"{run}.PASS.norm.vcf.gz"
    ann_vcf = ANN_DIR / f"{run}.annotated.vcf.gz"

    print(f"\nRun: {run}")

    # Check PASS VCF
    if not pass_vcf.exists():
        print(f"  [MISSING] Input PASS VCF: {pass_vcf}")
    else:
        print(f"  [EXISTS]  Input PASS VCF ({pass_vcf.stat().st_size} bytes)")

    # Check Annotated VCF
    if not ann_vcf.exists():
        print(f"  [MISSING] Annotated VCF:  {ann_vcf}")
    else:
        print(f"  [EXISTS]  Annotated VCF   ({ann_vcf.stat().st_size} bytes)")

        # Count records and check ANN tag
        record_count = 0
        has_ann = False

        try:
            with gzip.open(ann_vcf, "rt") as f:
                for line in f:
                    if not line.startswith("#"):
                        record_count += 1
                        if "ANN=" in line or "EFF=" in line:
                            has_ann = True
        except Exception as e:
            print(f"  [ERROR] Could not read VCF: {e}")

        print(f"  [METRICS] Variant records: {record_count} | Has ANN/EFF tag: {has_ann}")

File Status Check:

Run: SRR2589044
  [EXISTS]  Input PASS VCF (1744 bytes)
  [EXISTS]  Annotated VCF   (197 bytes)
  [METRICS] Variant records: 2 | Has ANN/EFF tag: False

Run: SRR2584863
  [MISSING] Input PASS VCF: results/vcf/SRR2584863.PASS.norm.vcf.gz
  [MISSING] Annotated VCF:  results/annotation/SRR2584863.annotated.vcf.gz

Run: SRR2584866
  [MISSING] Input PASS VCF: results/vcf/SRR2584866.PASS.norm.vcf.gz
  [MISSING] Annotated VCF:  results/annotation/SRR2584866.annotated.vcf.gz


In [ ]:
import gzip

vcf_path = "results/vcf/SRR2589044.PASS.norm.vcf.gz"

with gzip.open(vcf_path, "rt") as f:
    for line in f:
        if not line.startswith("#"):
            cols = line.strip().split("\t")
            print(f"VCF Chromosome ID: '{cols[0]}'")
            break

VCF Chromosome ID: 'CP000819.1'


In [ ]:
#!/usr/bin/env python3
from pathlib import Path
import pandas as pd

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------
INPUT = Path("results/statistics/annotation/all_annotated_variants.tsv")
OUT_DIR = Path("results/statistics/annotation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Read Input Data
# ----------------------------------------------------------
# Reading TSV with string types to preserve column indexing matching AWK ($1=col 0, $2=col 1, etc.)
df = pd.read_csv(INPUT, sep="\t", dtype=str)

# Map AWK 1-based column indices to Python 0-based column names/positions
# AWK: $2=col 1, $10=col 9, $11=col 10, $12=col 11
gen_col = df.columns[1]  # Generation ($2)
con_col = df.columns[9]  # Consequence ($10)
imp_col = df.columns[10]  # Impact ($11)
gene_col = df.columns[11]  # Gene ($12)

# Convert generation column to numeric for accurate numerical sorting (-k1,1n)
df["_gen_num"] = pd.to_numeric(df[gen_col], errors="coerce")

# ----------------------------------------------------------
# 1. Impact summary
# AWK: sort -t $'\t' -k1,1n -k2,2
# ----------------------------------------------------------
impact_summary = (
    df.groupby([gen_col, "_gen_num", imp_col], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(by=["_gen_num", imp_col], ascending=[True, True])
    [[gen_col, imp_col, "count"]]
    .rename(columns={gen_col: "generation", imp_col: "impact"})
)
impact_summary.to_csv(OUT_DIR / "impact_summary.tsv", sep="\t", index=False)

# ----------------------------------------------------------
# 2. Consequence summary
# AWK: sort -t $'\t' -k1,1n -k3,3nr
# ----------------------------------------------------------
consequence_summary = (
    df.groupby([gen_col, "_gen_num", con_col], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(by=["_gen_num", "count"], ascending=[True, False])
    [[gen_col, con_col, "count"]]
    .rename(columns={gen_col: "generation", con_col: "consequence"})
)
consequence_summary.to_csv(
    OUT_DIR / "consequence_summary.tsv", sep="\t", index=False
)

# ----------------------------------------------------------
# 3. Gene summary
# AWK: NR > 1 && $12 != "" ; sort -t $'\t' -k1,1n -k3,3nr
# ----------------------------------------------------------
df_gene_filtered = df[
    df[gene_col].notna() & (df[gene_col].str.strip() != "")
].copy()

gene_summary = (
    df_gene_filtered.groupby([gen_col, "_gen_num", gene_col], dropna=False)
    .size()
    .reset_index(name="annotations")
    .sort_values(by=["_gen_num", "annotations"], ascending=[True, False])
    [[gen_col, gene_col, "annotations"]]
    .rename(columns={gen_col: "generation", gene_col: "gene"})
)
gene_summary.to_csv(OUT_DIR / "gene_summary.tsv", sep="\t", index=False)

# ----------------------------------------------------------
# 4. High-impact variants
# AWK: NR==1 || $11=="HIGH"
# ----------------------------------------------------------
high_impact_df = df[df[imp_col] == "HIGH"].drop(columns=["_gen_num"])
# Retain original header and full column structure
high_impact_df.to_csv(
    OUT_DIR / "high_impact_variants.tsv", sep="\t", index=False
)

# ----------------------------------------------------------
# Terminal Output (Matches `column -t -s $'\t'` and `head -30`)
# ----------------------------------------------------------
print("Annotation summaries created.\n")

print("=== IMPACT ===")
print(impact_summary.to_string(index=False))

print("\n=== HIGH IMPACT ===")
print(high_impact_df.head(29).to_string(index=False))

Annotation summaries created.

=== IMPACT ===
Empty DataFrame
Columns: [generation, impact, count]
Index: []

=== HIGH IMPACT ===
Empty DataFrame
Columns: [sample, generation, run, chrom, position, ref, alt, qual, allele, consequence, impact, gene, gene_id, feature_type, feature_id, biotype, hgvs_c, hgvs_p]
Index: []


In [ ]:
#!/usr/bin/env python3
from pathlib import Path
import pandas as pd

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------
INPUT = Path("results/statistics/annotation/all_annotated_variants.tsv")
OUT_DIR = Path("results/statistics/annotation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Read Input Data
# ----------------------------------------------------------
# Reading TSV with string types to preserve column indexing matching AWK ($1=col 0, $2=col 1, etc.)
df = pd.read_csv(INPUT, sep="\t", dtype=str)

# Map AWK 1-based column indices to Python 0-based column names/positions
# AWK: $2=col 1, $10=col 9, $11=col 10, $12=col 11
gen_col = df.columns[1]  # Generation ($2)
con_col = df.columns[9]  # Consequence ($10)
imp_col = df.columns[10]  # Impact ($11)
gene_col = df.columns[11]  # Gene ($12)

# Convert generation column to numeric for accurate numerical sorting (-k1,1n)
df["_gen_num"] = pd.to_numeric(df[gen_col], errors="coerce")

# ----------------------------------------------------------
# 1. Impact summary
# AWK: sort -t $'\t' -k1,1n -k2,2
# ----------------------------------------------------------
impact_summary = (
    df.groupby([gen_col, "_gen_num", imp_col], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(by=["_gen_num", imp_col], ascending=[True, True])
    [[gen_col, imp_col, "count"]]
    .rename(columns={gen_col: "generation", imp_col: "impact"})
)
impact_summary.to_csv(OUT_DIR / "impact_summary.tsv", sep="\t", index=False)

# ----------------------------------------------------------
# 2. Consequence summary
# AWK: sort -t $'\t' -k1,1n -k3,3nr
# ----------------------------------------------------------
consequence_summary = (
    df.groupby([gen_col, "_gen_num", con_col], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(by=["_gen_num", "count"], ascending=[True, False])
    [[gen_col, con_col, "count"]]
    .rename(columns={gen_col: "generation", con_col: "consequence"})
)
consequence_summary.to_csv(
    OUT_DIR / "consequence_summary.tsv", sep="\t", index=False
)

# ----------------------------------------------------------
# 3. Gene summary
# AWK: NR > 1 && $12 != "" ; sort -t $'\t' -k1,1n -k3,3nr
# ----------------------------------------------------------
df_gene_filtered = df[
    df[gene_col].notna() & (df[gene_col].str.strip() != "")
].copy()

gene_summary = (
    df_gene_filtered.groupby([gen_col, "_gen_num", gene_col], dropna=False)
    .size()
    .reset_index(name="annotations")
    .sort_values(by=["_gen_num", "annotations"], ascending=[True, False])
    [[gen_col, gene_col, "annotations"]]
    .rename(columns={gen_col: "generation", gene_col: "gene"})
)
gene_summary.to_csv(OUT_DIR / "gene_summary.tsv", sep="\t", index=False)

# ----------------------------------------------------------
# 4. High-impact variants
# AWK: NR==1 || $11=="HIGH"
# ----------------------------------------------------------
high_impact_df = df[df[imp_col] == "HIGH"].drop(columns=["_gen_num"])
# Retain original header and full column structure
high_impact_df.to_csv(
    OUT_DIR / "high_impact_variants.tsv", sep="\t", index=False
)

# ----------------------------------------------------------
# Terminal Output (Matches `column -t -s $'\t'` and `head -30`)
# ----------------------------------------------------------
print("Annotation summaries created.\n")

print("=== IMPACT ===")
print(impact_summary.to_string(index=False))

print("\n=== HIGH IMPACT ===")
print(high_impact_df.head(29).to_string(index=False))

In [ ]:
#!/usr/bin/env python3
from pathlib import Path
import pandas as pd

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------
INPUT = Path("results/statistics/annotation/all_annotated_variants.tsv")
OUT_DIR = Path("results/statistics/annotation")
OUTPUT = OUT_DIR / "primary_variant_annotations.tsv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Severity impact priority mapping
SEVERITY = {"MODIFIER": 1, "LOW": 2, "MODERATE": 3, "HIGH": 4}

# ----------------------------------------------------------
# Process File
# ----------------------------------------------------------
# Read as string to prevent auto-type conversions altering coordinates/alleles
df = pd.read_csv(INPUT, sep="\t", dtype=str)

# Map AWK 1-based column indices to Python 0-based column positions:
# $1=Sample, $2=Generation, $4=CHROM, $5=POS, $6=REF, $7=ALT, $11=IMPACT
col_sample = df.columns[0]
col_gen = df.columns[1]
col_chrom = df.columns[3]
col_pos = df.columns[4]
col_ref = df.columns[5]
col_alt = df.columns[6]
col_impact = df.columns[10]

# Add numeric severity score for sorting/deduplication
df["_score"] = df[col_impact].map(SEVERITY).fillna(0)

# Deduplicate key: Sample | CHROM | POS | REF | ALT ($1|$4|$5|$6|$7)
dedup_keys = [col_sample, col_chrom, col_pos, col_ref, col_alt]

# Sort by severity score descending so drop_duplicates keeps the highest-impact entry
df_primary = (
    df.sort_values(by="_score", ascending=False)
    .drop_duplicates(subset=dedup_keys, keep="first")
    .copy()
)

# Convert generation ($2) and position ($5) to numeric for accurate sorting (-k2,2n -k5,5n)
df_primary["_gen_num"] = pd.to_numeric(df_primary[col_gen], errors="coerce")
df_primary["_pos_num"] = pd.to_numeric(df_primary[col_pos], errors="coerce")

# Sort by generation numerically, then position numerically
df_primary = df_primary.sort_values(
    by=["_gen_num", "_pos_num"], ascending=[True, True]
)

# Drop helper temporary columns before output
df_output = df_primary.drop(columns=["_score", "_gen_num", "_pos_num"])

# Save primary annotation output
df_output.to_csv(OUTPUT, sep="\t", index=False)

# ----------------------------------------------------------
# Terminal Output (Matches `column -t -s $'\t'` and `head -30`)
# ----------------------------------------------------------
print("Primary annotation table created:")
print(f"{OUTPUT}\n")

# Print top 29 data rows + header (matches bash `head -30`)
print(df_output.head(29).to_string(index=False))

In [ ]:
#!/usr/bin/env python3
from pathlib import Path
import shutil
import subprocess
import sys
import time
import pandas as pd

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------
METADATA_PATH = Path("metadata/samples.tsv")
REF = Path("snpeff/data/REL606/genes.gbk")

TRIM_DIR = Path("data/trimmed")
UNPAIRED_DIR = TRIM_DIR / "unpaired"

OUT_DIR = Path("results/breseq")
GD_DIR = Path("results/gd")

THREADS = 4

# Create output directories
OUT_DIR.mkdir(parents=True, exist_ok=True)
GD_DIR.mkdir(parents=True, exist_ok=True)

print("==============================================")
print("breseq population/structural analysis")
print(f"Started: {time.ctime()}")
print(f"Reference: {REF}")
print("==============================================")

# ----------------------------------------------------------
# Pre-check Reference Genome
# ----------------------------------------------------------
if not REF.exists() or REF.stat().st_size == 0:
    print(f"ERROR: GenBank reference missing:\n{REF}", file=sys.stderr)
    sys.exit(1)

# Find breseq binary
breseq_bin = shutil.which("breseq") or "breseq"

# ----------------------------------------------------------
# Load Metadata and Process Samples
# ----------------------------------------------------------
df_samples = pd.read_csv(METADATA_PATH, sep="\t")
df_samples.columns = df_samples.columns.str.upper()

for _, row in df_samples.iterrows():
    sample = str(row["SAMPLE"])
    generation = str(row["GENERATION"])
    run = str(row["RUN"])

    print("\n==============================================")
    print(f"Sample:     {sample}")
    print(f"Generation: {generation}")
    print(f"Run:        {run}")
    print("==============================================")

    r1 = TRIM_DIR / f"{run}_1.trim.fastq.gz"
    r2 = TRIM_DIR / f"{run}_2.trim.fastq.gz"
    u1 = UNPAIRED_DIR / f"{run}_1.unpaired.fastq.gz"
    u2 = UNPAIRED_DIR / f"{run}_2.unpaired.fastq.gz"

    sample_out = OUT_DIR / run
    gd_file = sample_out / "output" / "output.gd"
    target_gd = GD_DIR / f"{run}.gd"

    # Skip if output GenomeDiff (.gd) file already exists and is non-empty
    if gd_file.exists() and gd_file.stat().st_size > 0:
        print(f"[SKIP] Existing breseq result:\n{gd_file}")
    else:
        # Build read arguments
        reads = [str(r1), str(r2)]

        # Include surviving orphan reads if present
        if u1.exists() and u1.stat().st_size > 0:
            reads.append(str(u1))

        if u2.exists() and u2.stat().st_size > 0:
            reads.append(str(u2))

        # Construct breseq command
        cmd = [
            breseq_bin,
            "-j",
            str(THREADS),
            "-p",  # Polymorphic mode
            "-n",
            sample,
            "-r",
            str(REF),
            "-o",
            str(sample_out),
            *reads,
        ]

        print(f"Running breseq for {run}...")
        res = subprocess.run(cmd)

        if res.returncode != 0:
            print(
                f"ERROR: breseq execution failed for {run}", file=sys.stderr
            )
            sys.exit(1)

    # Confirm GenomeDiff file was created
    if not gd_file.exists() or gd_file.stat().st_size == 0:
        print(f"ERROR: breseq did not produce:\n{gd_file}", file=sys.stderr)
        sys.exit(1)

    # Copy output GD file to centralized results/gd/ directory
    shutil.copy2(gd_file, target_gd)
    print(f"[OK] {run}")

# ----------------------------------------------------------
# Final Summary Output
# ----------------------------------------------------------
print("\n==============================================")
print("breseq runs complete")
print(f"Finished: {time.ctime()}")
print("==============================================")

In [ ]:
#!/usr/bin/env python3
from pathlib import Path
import shutil
import subprocess
import sys
import time
import pandas as pd

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------
METADATA_PATH = Path("metadata/samples.tsv")
REF = Path("snpeff/data/REL606/genes.gbk")

TRIM_DIR = Path("data/trimmed")
UNPAIRED_DIR = TRIM_DIR / "unpaired"

OUT_DIR = Path("results/breseq_consensus")
GD_DIR = Path("results/gd_consensus")

THREADS = 4

# Create output directories
OUT_DIR.mkdir(parents=True, exist_ok=True)
GD_DIR.mkdir(parents=True, exist_ok=True)

print("==============================================")
print("breseq consensus analysis of Ara-3 clones")
print(f"Started: {time.ctime()}")
print("==============================================")

# ----------------------------------------------------------
# Pre-check Reference Genome
# ----------------------------------------------------------
if not REF.exists() or REF.stat().st_size == 0:
    print(f"ERROR: GenBank reference missing:\n{REF}", file=sys.stderr)
    sys.exit(1)

# Find breseq binary
breseq_bin = shutil.which("breseq") or "breseq"

# ----------------------------------------------------------
# Load Metadata and Process Samples
# ----------------------------------------------------------
df_samples = pd.read_csv(METADATA_PATH, sep="\t")
df_samples.columns = df_samples.columns.str.upper()

for _, row in df_samples.iterrows():
    sample = str(row["SAMPLE"])
    generation = str(row["GENERATION"])
    run = str(row["RUN"])

    print("\n----------------------------------------------")
    print(f"Sample:     {sample}")
    print(f"Generation: {generation}")
    print(f"Run:        {run}")
    print("----------------------------------------------")

    r1 = TRIM_DIR / f"{run}_1.trim.fastq.gz"
    r2 = TRIM_DIR / f"{run}_2.trim.fastq.gz"
    u1 = UNPAIRED_DIR / f"{run}_1.unpaired.fastq.gz"
    u2 = UNPAIRED_DIR / f"{run}_2.unpaired.fastq.gz"

    sample_out = OUT_DIR / run
    gd_file = sample_out / "output" / "output.gd"
    target_gd = GD_DIR / f"{run}.gd"

    # Build read arguments
    reads = [str(r1), str(r2)]

    # Include surviving orphan reads if present
    if u1.exists() and u1.stat().st_size > 0:
        reads.append(str(u1))

    if u2.exists() and u2.stat().st_size > 0:
        reads.append(str(u2))

    # Skip if output GenomeDiff (.gd) file already exists and is non-empty
    if gd_file.exists() and gd_file.stat().st_size > 0:
        print(f"[SKIP] Existing result for {run}")
    else:
        # Construct breseq consensus command (omitting -p)
        cmd = [
            breseq_bin,
            "-j",
            str(THREADS),
            "-n",
            sample,
            "-r",
            str(REF),
            "-o",
            str(sample_out),
            *reads,
        ]

        print(f"Running breseq consensus for {run}...")
        res = subprocess.run(cmd)

        if res.returncode != 0:
            print(
                f"ERROR: breseq execution failed for {run}", file=sys.stderr
            )
            sys.exit(1)

    # Confirm GenomeDiff file was created
    if not gd_file.exists() or gd_file.stat().st_size == 0:
        print(f"ERROR: breseq did not produce:\n{gd_file}", file=sys.stderr)
        sys.exit(1)

    # Copy output GD file to centralized results/gd_consensus/ directory
    shutil.copy2(gd_file, target_gd)

print("\nbreseq consensus analysis complete.")